In [1]:
url <- "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2025-26/gws/merged_gw.csv"
df <- read.csv(url)


In [2]:
install.packages("zoo")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [14]:
df <- df %>%
  arrange(element, GW) %>%
  group_by(element) %>%
  mutate(
    pts_lag1 = lag(total_points, 1),
    pts_lag2 = lag(total_points, 2),
    pts_lag3 = lag(total_points, 3),
    total_points_roll3 = rowMeans(cbind(pts_lag1, pts_lag2, pts_lag3), na.rm = TRUE),
    min_lag1 = lag(minutes, 1),
    min_lag2 = lag(minutes, 2),
    min_lag3 = lag(minutes, 3),
    minutes_roll3 = rowMeans(cbind(min_lag1, min_lag2, min_lag3), na.rm = TRUE),
    ict_lag1 = lag(ict_index, 1),
    ict_lag2 = lag(ict_index, 2),
    ict_lag3 = lag(ict_index, 3),
    ict_index_roll3 = rowMeans(cbind(ict_lag1, ict_lag2, ict_lag3), na.rm = TRUE),
    bps_lag1 = lag(bps, 1),
    bps_lag2 = lag(bps, 2),
    bps_lag3 = lag(bps, 3),
    bps_roll3 = rowMeans(cbind(bps_lag1, bps_lag2, bps_lag3), na.rm = TRUE),
    xg_lag1 = lag(expected_goal_involvements, 1),
    xg_lag2 = lag(expected_goal_involvements, 2),
    xg_lag3 = lag(expected_goal_involvements, 3),
    expected_goal_involvements_roll3 = rowMeans(cbind(xg_lag1, xg_lag2, xg_lag3), na.rm = TRUE)
  ) %>%
  ungroup()

In [16]:
df <- df %>%
  arrange(element, GW) %>%
  group_by(element) %>%
  mutate(
    target_next_points = lead(total_points, 1)
  ) %>%
  ungroup()

In [19]:
install.packages("randomForest")
library(randomForest)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.


Attaching package: ‘randomForest’


The following object is masked from ‘package:dplyr’:

    combine




In [23]:
rf_model <- randomForest(
  target_next_points ~ total_points_roll3 + minutes_roll3 + ict_index_roll3 +
                       bps_roll3 + expected_goal_involvements_roll3,
  data = train_data,
  ntree = 300,
  mtry = 3,
  importance = TRUE
)

rf_model


Call:
 randomForest(formula = target_next_points ~ total_points_roll3 +      minutes_roll3 + ict_index_roll3 + bps_roll3 + expected_goal_involvements_roll3,      data = train_data, ntree = 300, mtry = 3, importance = TRUE) 
               Type of random forest: regression
                     Number of trees: 300
No. of variables tried at each split: 3

          Mean of squared residuals: 4.356502
                    % Var explained: 22.64

In [26]:
latest_form <- model_df %>%
  group_by(element) %>%
  filter(GW == max(GW)) %>%
  ungroup()

nrow(latest_form)

[1] 838

In [28]:
current_url <- "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2026-27/players_raw.csv"
current_players <- read.csv(current_url)

current_players$full_name <- paste(current_players$first_name, current_players$second_name)

head(current_players %>% select(full_name, now_cost, element_type, team))

,full_name,now_cost,element_type,team
,<chr>,<int>,<int>,<int>
1,David Raya Martín,60,1,1
2,Kepa Arrizabalaga Revuelta,50,1,1
3,Illan Meslier,50,1,1
4,Gabriel dos Santos Magalhães,80,2,1
5,Jurriën Timber,65,2,1
6,William Saliba,60,2,1


In [36]:
merged <- merged %>%
  mutate(position_2026 = case_when(
    element_type == 1 ~ "GK",
    element_type == 2 ~ "DEF",
    element_type == 3 ~ "MID",
    element_type == 4 ~ "FWD"
  ))

merged %>% count(position_2026)

position_2026,n
<chr>,<int>
DEF,148
FWD,52
GK,47
MID,203


In [37]:
install.packages("lpSolve")
library(lpSolve)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [40]:
opt_data <- merged %>%
  distinct(name, .keep_all = TRUE) %>%
  filter(!is.na(predicted_gw1_points), !is.na(now_cost), !is.na(position_2026), !is.na(team.y)) %>%
  rename(team = team.y)

n <- nrow(opt_data)
cat("Players available for selection:", n, "\n")

teams_list <- unique(opt_data$team)
cat("Number of distinct clubs:", length(teams_list), "\n")

Players available for selection: 450 
Number of distinct clubs: 20 


In [41]:
n <- nrow(opt_data)

objective <- opt_data$predicted_gw1_points

cost_constraint <- opt_data$now_cost

pos_constraints <- sapply(c("GK", "DEF", "MID", "FWD"), function(p) as.numeric(opt_data$position_2026 == p))

team_constraints <- sapply(teams_list, function(t) as.numeric(opt_data$team == t))

constraint_matrix <- rbind(
  cost_constraint,
  t(pos_constraints),
  t(team_constraints)
)

constraint_dir <- c(
  "<=",
  "==", "==", "==", "==",
  rep("<=", length(teams_list))
)

constraint_rhs <- c(
  1000,
  2, 5, 5, 3,
  rep(3, length(teams_list))
)

result <- lp(
  direction = "max",
  objective.in = objective,
  const.mat = constraint_matrix,
  const.dir = constraint_dir,
  const.rhs = constraint_rhs,
  all.bin = TRUE
)

result$status

[1] 0

In [43]:
selected_indices <- which(result$solution == 1)
optimal_squad <- opt_data[selected_indices, ] %>%
  select(name, position_2026, team, now_cost, predicted_gw1_points) %>%
  arrange(match(position_2026, c("GK", "DEF", "MID", "FWD")), desc(predicted_gw1_points))

optimal_squad$now_cost <- optimal_squad$now_cost / 10

print(optimal_squad, n = 15)

# A tibble: 15 × 5
   name                      position_2026  team now_cost predicted_gw1_points
   <chr>                     <chr>         <int>    <dbl>                <dbl>
 1 David Raya Martín         GK                1      6                   5.68
 2 Emiliano Martínez Romero  GK                2      5                   4.41
 3 Adrien Truffert           DEF               3      5.5                 4.90
 4 Marc Guéhi                DEF              15      6                   4.58
 5 Riccardo Calafiori        DEF               1      5.5                 4.47
 6 James Tarkowski           DEF               9      6                   4.44
 7 Nathan Collins            DEF               4      5.5                 4.36
 8 Saša Lukić                MID              10      5                   5.88
 9 Enzo Le Fée               MID              20      6                   5.43
10 Sandro Tonali             MID              19      5.5                 5.41
11 Jack Hinshelwood          MID 

In [44]:
total_cost <- sum(optimal_squad$now_cost)
total_points <- sum(optimal_squad$predicted_gw1_points)
cat("Total cost:", total_cost, "million\n")
cat("Total predicted GW1 points:", total_points, "\n")

Total cost: 96 million
Total predicted GW1 points: 74.74088 
